In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [3]:
# Loading data


In [4]:
df  = pd.read_excel('boom_english.xlsx')
df.head(5)
df.tail(10)

,website,link,unique_id,title,publish_date,content,top_image,image_links,links_in_text,bold_text,tweet_id,video,topic,tags,claim,investigation
899,boom_english,https://www.boomlive.in/from-a-photo-of-indian...,from-a-photo-of-indian-soldiers-at-siachen-to-...,From A Photo Of Indian Soldiers At Siachen To ...,23-12-2017 03:03:00,"[""In this week's News You Almost Believed we s...",NaN,['https://www.boomlive.in/wp-content/uploads/2...,['https://www.boomlive.in/wp-content/uploads/2...,"[None, 'here', None, None, 'pic.twitter.com/7Q...",['943692045759188992'],NaN,['Fake News » '],"['Amit Malviya', 'fake news', 'featured', 'Ind...",Claim 1 - A photo showing two soldiers bravin...,Investigation 1 - Despite numerous replies poi...
900,boom_english,https://www.boomlive.in/health/does-drinking-a...,does-drinking-alcohol-prevent-coronavirus-6935,Does Drinking Alcohol Prevent Coronavirus?,19-02-2020 13:15:00,['Shiv Sena mouth piece suggests that drinking...,https://www.boomlive.in/h-upload/2020/02/19/91...,NaN,['http://hindisaamana.4cplus.net/imageview_160...,"['here', 'Can Avoiding Ice Creams And Cold Dri...",['1222774654596734976'],['https://www.facebook.com/plugins/post.php?hr...,['Health » '],"['Coronavirus India', 'Corona latest', 'Corona...",Shiv Sena mouth piece suggests that drinking a...,Fact Check The article mentions that researche...
901,boom_english,https://www.boomlive.in/no-russian-president-v...,no-russian-president-vladimir-putin-is-not-att...,"No, Russian President Vladimir Putin Is Not At...",30-10-2019 13:24:00,['OpIndia and MyNation missreported that Russi...,https://www.boomlive.in/wp-content/uploads/201...,['https://www.boomlive.in/wp-content/uploads/2...,"['http://archive.is/kZt7R', 'http://archive.is...","['One India Tamil', 'OpIndia', 'My India', '#J...","['1189095772437921792', '1189095772437921792',...",NaN,['Fake News » '],"['bull taming sport', 'Fact Check', 'fake news...",OpIndia and MyNation missreported that Russian...,Modi is putting tamil nadu and its culture on ...
902,boom_english,https://www.boomlive.in/netizens-misidentify-s...,netizens-misidentify-student-photographed-with...,Netizens Misidentify Student Photographed With...,21-09-2019 10:27:00,"['BOOM got in touch with Shilpi Afreen, also a...",https://www.boomlive.in/wp-content/uploads/201...,['https://www.boomlive.in/wp-content/uploads/2...,['http://archive.is/XmqHP'],['posts'],NaN,['https://www.facebook.com/plugins/post.php?hr...,['FactCheck » '],"['ABVP', 'Bharatiya Janata Party', 'fake news'...","BOOM got in touch with Shilpi Afreen, also a ...",Fact Check BOOM could ascertain that the woma...
903,boom_english,https://www.boomlive.in/the-hindu-retracts-dyi...,the-hindu-retracts-dying-woman-molested-video-...,"The Hindu Retracts 'Dying Woman Molested, Vide...",04-10-2017 07:02:00,['The Hindu apologises for its story. Says the...,NaN,['https://www.boomlive.in/wp-content/uploads/2...,['https://www.firstpost.com/india/elphinstone-...,"['Firstpost', 'IndiaTimes', 'ScoopWhoop', 'New...","['914571752885280768', '915270586506424320', '...",['https://www.facebook.com/plugins/post.php?hr...,['Fake News » '],"['Elphinstone Road Station', 'featured', 'Jour...",The Hindu apologises for its story. Says the a...,BOOM was able to find a cached version of the ...
904,boom_english,https://www.boomlive.in/pak-scribe-shares-old-...,pak-scribe-shares-old-incident-of-indian-force...,Pak Scribe Shares Old Incident Of Indian Force...,17-08-2019 19:24:00,['BOOM found that the video tweeted by Mir was...,https://www.boomlive.in/wp-content/uploads/201...,['https://www.boomlive.in/wp-content/uploads/2...,"['https://t.co/MeetCzzXmO', 'http://archive.is...","['pic.twitter.com/MeetCzzXmO', 'here', 'video'...",['1162729926111506434'],['https://www.facebook.com/plugins/video.php?h...,['Fake News » '],"['featured', 'Hamid Mir', 'Human shield', 'Ind...",BOOM found that the video tweeted by Mir was f...,FACT-CHECK As several Twitter users pointed ou...
905,boom_english,https://

In [5]:
df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 909 entries, 0 to 908
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   website        909 non-null    object
 1   link           909 non-null    object
 2   unique_id      909 non-null    object
 3   title          909 non-null    object
 4   publish_date   909 non-null    object
 5   content        909 non-null    object
 6   top_image      678 non-null    object
 7   image_links    611 non-null    object
 8   links_in_text  861 non-null    object
 9   bold_text      861 non-null    object
 10  tweet_id       471 non-null    object
 11  video          481 non-null    object
 12  topic          909 non-null    object
 13  tags           902 non-null    object
 14  claim          908 non-null    object
 15  investigation  909 non-null    object
dtypes: object(16)
memory usage: 113.8+ KB


In [6]:
df.shape

(909, 16)

## Label Extraction

In [7]:

import ast


def safe_parse_list(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [x]  # fallback if it's not actually list-formatted
        

df['topic_parsed'] = df['topic'].apply(safe_parse_list)
df['tags_parsed'] = df['tags'].apply(safe_parse_list)


# --- Step 2: Flatten topic into a single string per row for easy matching ---
df['topic_flat'] = df['topic_parsed'].apply(lambda lst: ' '.join(lst) if lst else '')

# --- Step 3: Check what topic categories actually exist ---
print(df['topic_flat'].value_counts())


topic_flat
Fake News »                     669
World »                          90
FactCheck »                      40
Fast Check »                     34
World »  Coronavirus »           29
Health »                         14
Videos »  Fact Vs Fiction »       6
Fact File »                       6
Entertainment »                   4
Politics »  Elections »           4
Coronavirus »                     3
Business »                        3
India »                           2
Opinion »                         1
Fake News »  Mob Lynching »       1
Politics »                        1
Coronavirus News »                1
AFP »                             1
Name: count, dtype: int64


In [8]:

def derive_label(topic_str):
    topic_str = topic_str.lower()
    if 'fake news' in topic_str:
        return 'fake'
    elif 'factcheck' in topic_str:
        return 'factcheck_ambiguous'   # needs manual/secondary check
    else:
        return 'other'  



df['label_v1'] = df['topic_flat'].apply(derive_label)
print(df['label_v1'].value_counts())



# --- Step 5: Spot-check the ambiguous 'factcheck' rows manually ---
pd.set_option('display.max_colwidth', 150)

df[df['label_v1'] == 'fake'][['link' ,'title', 'claim', 'investigation']].head(5)

label_v1
fake                   670
other                  199
factcheck_ambiguous     40
Name: count, dtype: int64


,link,title,claim,investigation
0,https://www.boomlive.in/fake-news/video-of-daring-bird-rescue-in-a-chopper-is-not-from-surat-6895,Video Of Daring Bird Rescue In A Chopper Is Not From Surat,"The video is originally from Virginia Beach in US, where a seagull was rescued from an overhead wire A heartwarming video of a seagull rescue oper...","Fact Check BOOM ran a relevant keyword search on YouTube with words like 'Bird rescued by helicopter' and found the same video, that was first u..."
2,https://www.boomlive.in/fake-news/fake-nita-ambanis-tweet-supporting-pm-modi-amit-shah-6366,"FAKE: Nita Ambani's 'Tweet' Supporting PM Modi, Amit Shah",Twitter has since suspended the fake account impersonating Nita Ambani A tweet by a fake Nita Ambani account urging people to support and stand by...,We also looked for media reports about any comment by Nita Ambani about the Citizenship bill and did not find any such reports. Searching with ...
3,https://www.boomlive.in/fake-news/image-of-cop-attacked-in-up-shared-with-communal-spin-in-west-bengal-7733,Image Of Cop Attacked In UP Shared With Communal Spin In West Bengal,"BOOM found that the incident happened on June, 2017, in Kanpur where a clash broke out between agitators and police after a teen was raped in a ho...",Fact Check BOOM ran a reverse image search and found that the incident is neither related to West Bengal nor did it happen during the lockdown p...
4,https://www.boomlive.in/fake-news/no-video-does-not-show-street-vendors-evicted-before-trumps-gujarat-visit-6929,"No, Video Does Not Show Street Vendors Evicted Before Trump's Gujarat Visit",BOOM found that the video is not connected to Donald Trump's visit. A video of an eviction drive from Odisha is being falsely shared as street sid...,Fact Check BOOM had previously debunked the same video being shared as Uttar Pradesh in light of the protests against the Citizenship Amendment Ac...
5,https://www.boomlive.in/fake-news/video-of-migrant-shipwreck-off-libyan-coast-revived-with-coronavirus-spin-7585,Video Of Migrant Shipwreck Off Libyan Coast Revived With Coronavirus Spin,BOOM found that the video is from the Libyan coast where bodies were washed ashore from a shipwreck in 2014 A disturbing video showing the bodies ...,Fact Check BOOM reverse searched a few of the keyframes of the video and found that it is from a boat capsizing incident that took place off the ...


In [9]:
df[df['label_v1'] == 'factcheck_ambiguous'][['title', 'claim', 'investigation']].head(5)

,title,claim,investigation
23,Johnny Depp's Jack Sparrow Inspired By Krishna? News Sites Repeat Mystery Quote,More than ten websites carry story on #JackSparrowIsKrishna quoting Pirates of the Caribbean screenwriter. But BOOM finds no original source of th...,BOOM has reached out to TOI for the original source of the quote attributed to Ted Elliot and also tweeted to them. We are yet to receive a repl...
109,From A Texas Mosque Turning Away Hurricane Evacuees To Gauri Lankesh 'Patrick': Fake News This Week,Claim 1 - Fake news did not spare slain journalist Gauri Lankesh. Barely 48 hours after the firebrand journalist was shot dead outside her home by...,"Investigation 1 - However, a fact check revealed that the fake news brigade turned 'Patrike' meaning magazine in Kannada to 'Patrick'. Gauri Lank..."
154,"""Pure, Utter Rubbish,"" Says Raghuram Rajan About Fake Quote On PNB Scam","In an exclusive quote to BOOM, Raghuram Rajan rubbishes the fake quote circulating on social media about his alert on PNB scam File Pic: Raghuram ...",BOOM has not been independently able to verify who created this fake quote though many users have shared the image and the text with Postcard News...
192,Fake Ram Nath Kovind Twitter Accounts Get Busy,"BOOM that he is not present on social media yet. While some of the handles are fan or parody accounts, the description of some of the other fak...",BOOM had earlier followed @iRamnathKovind to keep track of its tweets and when @KatrinaKaifPost surfaced it already showed us as following the acc...
199,Did Three Nobel Winners Say Cancer Can Be Cured Without Medicine?,Hyderabad physiotherapist misinterprets findings of Nobel prize winners. A viral image stating that three Nobel Prize winners concluded that canc...,FactCheck: Autophagy means self-eating. Cells when starved of nutrients tend to search for damaged cells and consume them. While fas...


## Synthetic Data for user names (completely vaild as user are randomly initialized)

In [10]:
np.random.seed(42)  
 

df['final_label'] = df['label_v1'].replace({'factcheck_ambiguous': 'other'})
df['label_binary'] = df['final_label'].map({'fake': 'fake', 'other': 'real'})
 

first_names = [
    "Rohan", "Ananya", "Karan", "Priya", "Vikram", "Simran", "Arjun", "Neha",
    "Aditya", "Divya", "Rahul", "Ishita", "Kunal", "Meera", "Sameer", "Tanya",
    "Nikhil", "Pooja", "Aman", "Riya", "Varun", "Anjali", "Siddharth", "Kavya",
    "Manav", "Shreya", "Yash", "Nandini", "Rohit", "Aisha"
]
last_names = [
    "Mehta", "Iyer", "Malhotra", "Nair", "Sethi", "Kaur", "Rao", "Bhatt",
    "Kapoor", "Menon", "Verma", "Sharma", "Joshi", "Pillai", "Khan", "Chawla",
    "Das", "Reddy", "Gupta", "Choudhary", "Saxena", "Rana", "Mishra", "Iyengar",
    "Bose", "Agarwal", "Trivedi", "Pandey", "Kulkarni", "Fernandes"
]
 
all_combos = [f"{f} {l}" for f in first_names for l in last_names]
np.random.shuffle(all_combos)
user_pool = all_combos[:150]  



df['user'] = np.random.choice(user_pool, size=len(df), replace=True)

user_stats = df.groupby('user').agg(
    total_posts=('user', 'count'),
    fake_posts=('label_binary', lambda x: (x == 'fake').sum()),
    real_posts=('label_binary', lambda x: (x == 'real').sum())
).reset_index()

user_stats['fake_ratio'] = user_stats['fake_posts'] / user_stats['total_posts']


## Fake ratio by randomness

In [11]:
user_stats = user_stats.sort_values('total_posts', ascending=False)
print("Most active users overall:")
print(user_stats.head(10))
 
print("\nUsers with high fake_ratio AND reasonable activity (candidate 'X' suspicious group):")
print(user_stats[(user_stats['fake_ratio'] > 0.7) & (user_stats['total_posts'] >= 3)].sort_values('fake_ratio', ascending=False))
 
print("\nUsers with very low activity (candidate 'Z' inactive group):")
print(user_stats[user_stats['total_posts'] == 1].shape[0], "users posted only once")

Most active users overall:
                user  total_posts  fake_posts  real_posts  fake_ratio
63       Meera Mehta           13          10           3    0.769231
24   Arjun Fernandes           12           8           4    0.666667
19   Anjali Kulkarni           12          12           0    1.000000
109    Rohan Iyengar           12          10           2    0.833333
57   Manav Choudhary           11          10           1    0.909091
118    Sameer Sharma           11           5           6    0.454545
107      Riya Pillai           11           6           5    0.545455
7         Aisha Rana           10           9           1    0.900000
140     Vikram Mehta           10           8           2    0.800000
130    Simran Chawla           10          10           0    1.000000

Users with high fake_ratio AND reasonable activity (candidate 'X' suspicious group):
                user  total_posts  fake_posts  real_posts  fake_ratio
127  Siddharth Menon            6           6  

In [12]:
df.head(2)

,website,link,unique_id,title,publish_date,content,top_image,image_links,links_in_text,bold_text,...,tags,claim,investigation,topic_parsed,tags_parsed,topic_flat,label_v1,final_label,label_binary,user
0,boom_english,https://www.boomlive.in/fake-news/video-of-daring-bird-rescue-in-a-chopper-is-not-from-surat-6895,video-of-daring-bird-rescue-in-a-chopper-is-not-from-surat-6895,Video Of Daring Bird Rescue In A Chopper Is Not From Surat,14-02-2020 13:45:00,"['The video is originally from Virginia Beach in US, where a seagull was rescued from an overhead wire', 'A heartwarming video of a seagull rescue...",https://www.boomlive.in/h-upload/2020/02/14/823919-seagull-recues-virginia-beach.jpg,NaN,"['https://t.co/kJ9A8a7bx0', 'https://twitter.com/RatanKAgrawal/status/1227921905199403008', 'http://archive.is/CQtb0', 'https://www.facebook.com/s...","['pic.twitter.com/kJ9A8a7bx0', 'here', 'here', 'viral', 'The Mirror ', 'the Telegraph', 'Metro', 'False: 40 Out Of 62 AAP MLAs Accused Of Rape']",...,"['Seagull', 'Bird Rescue', 'Helicopter', 'Jain Samaj', 'Surat', 'Viral Video', 'Virginia']","The video is originally from Virginia Beach in US, where a seagull was rescued from an overhead wire A heartwarming video of a seagull rescue oper...","Fact Check BOOM ran a relevant keyword search on YouTube with words like 'Bird rescued by helicopter' and found the same video, that was first u...",[Fake News » ],"[Seagull, Bird Rescue, Helicopter, Jain Samaj, Surat, Viral Video, Virginia]",Fake News »,fake,fake,fake,Aisha Sethi
1,boom_english,https://www.boomlive.in/world/false-italian-police-forcibly-arrested-a-man-for-breaking-lockdown-7741,false-italian-police-forcibly-arrested-a-man-for-breaking-lockdown-7741,False: Italian Police Forcibly Arrested A Man For Breaking Lockdown,20-04-2020 09:42:00,"['The video is from Brazil where police arrested an intoxicated man for threatening officers with a knife.', 'The video was published on Facebook ...",https://www.boomlive.in/h-upload/2020/04/20/920021-italian-police.jpg,"['https://www.boomlive.in/h-upload/2020/03/31/918797-2020033115-ialy-beat.jpg', 'https://www.boomlive.in/h-upload/2020/03/31/918798-2020033115-glo...","['https://perma.cc/M9LV-GWZ3', 'https://www.boomlive.in/world/saddam-hussein-video-doctored-to-include-coronavirus-reference-7740', 'https://www.a...","['here ', 'ALSO READ: Saddam Hussein video doctored to include coronavirus reference', 'here', 'here', 'here ', 'here', 'here', 'here', 'here', 'h...",...,['COVID-19'],The video is from Brazil where police arrested an intoxicated man for threatening officers with a knife. The video was published on Facebook here...,"The claim is false. A reverse Google image search using video keyframes extracted with InVID, a video verification tool, and subsequent keyword se...",[World » ],[COVID-19],World »,other,other,real,Pooja Rana


In [13]:
import pandas as pd
import numpy as np

np.random.seed(42)  # reproducibility

# --- Step 1: Finalize binary label (unchanged from before) ---
df['final_label'] = df['label_v1'].replace({'factcheck_ambiguous': 'other'})
df['label_binary'] = df['final_label'].map({'fake': 'fake', 'other': 'real'})

# --- Step 2: Generate a pool of ~150 unique synthetic names ---
first_names = [
    "Rohan", "Ananya", "Karan", "Priya", "Vikram", "Simran", "Arjun", "Neha",
    "Aditya", "Divya", "Rahul", "Ishita", "Kunal", "Meera", "Sameer", "Tanya",
    "Nikhil", "Pooja", "Aman", "Riya", "Varun", "Anjali", "Siddharth", "Kavya",
    "Manav", "Shreya", "Yash", "Nandini", "Rohit", "Aisha"
]
last_names = [
    "Mehta", "Iyer", "Malhotra", "Nair", "Sethi", "Kaur", "Rao", "Bhatt",
    "Kapoor", "Menon", "Verma", "Sharma", "Joshi", "Pillai", "Khan", "Chawla",
    "Das", "Reddy", "Gupta", "Choudhary", "Saxena", "Rana", "Mishra", "Iyengar",
    "Bose", "Agarwal", "Trivedi", "Pandey", "Kulkarni", "Fernandes"
]

all_combos = [f"{f} {l}" for f in first_names for l in last_names]
np.random.shuffle(all_combos)
user_pool = all_combos[:150]  # pick 150 unique names out of 900 possible combos

# --- Step 3: Assign users to rows ---
# Background: ~95% of the user pool posts uniformly at random, regardless of label
# (this represents normal, non-coordinated behavior)
#
# Injected coordinated minority: a small group (~5-8 users) is deliberately given a
# much higher chance of landing on FAKE-labelled rows only. This simulates a real
# coordinated propaganda cluster. NOTE: this is a disclosed synthetic limitation --
# FactDrill has no real user-propagation data, so a true anomaly signal must be
# injected for the demo. State this openly in your slides.

n_coordinated = 7
coordinated_users = user_pool[:n_coordinated]
background_users = user_pool[n_coordinated:]

def assign_user(row_label):
    # 8% chance any given fake row is hit by the coordinated cluster
    if row_label == 'fake' and np.random.rand() < 0.35:
        return np.random.choice(coordinated_users)
    else:
        return np.random.choice(background_users)

df['user'] = df['label_binary'].apply(assign_user)

# --- Step 4: Post-hoc analysis (this is where "suspicious" should emerge from, not be built in) ---
user_stats = df.groupby('user').agg(
    total_posts=('user', 'count'),
    fake_posts=('label_binary', lambda x: (x == 'fake').sum()),
    real_posts=('label_binary', lambda x: (x == 'real').sum())
).reset_index()
user_stats['fake_ratio'] = user_stats['fake_posts'] / user_stats['total_posts']

user_stats = user_stats.sort_values('total_posts', ascending=False)
print("Most active users overall:")
print(user_stats.head(10))

print("\nUsers with high fake_ratio AND reasonable activity (candidate 'X' suspicious group):")
print(user_stats[(user_stats['fake_ratio'] > 0.7) & (user_stats['total_posts'] >= 15)].sort_values('fake_ratio', ascending=False))

print("\nUsers with very low activity (candidate 'Z' inactive group):")
print(user_stats[user_stats['total_posts'] == 1].shape[0], "users posted only once")

Most active users overall:
                 user  total_posts  fake_posts  real_posts  fake_ratio
16       Ananya Menon           45          45           0    1.000000
69      Nandini Reddy           42          42           0    1.000000
59       Manav Sharma           37          37           0    1.000000
100        Riya Gupta           34          34           0    1.000000
28    Divya Fernandes           33          33           0    1.000000
78          Neha Rana           28          28           0    1.000000
47        Karan Verma           21          21           0    1.000000
73          Neha Iyer            9           6           3    0.666667
109  Sameer Fernandes            9           5           4    0.555556
77          Neha Nair            9           7           2    0.777778

Users with high fake_ratio AND reasonable activity (candidate 'X' suspicious group):
                user  total_posts  fake_posts  real_posts  fake_ratio
16      Ananya Menon           45   

In [14]:
user_stats.head(50)

,user,total_posts,fake_posts,real_posts,fake_ratio
16,Ananya Menon,45,45,0,1.000000
69,Nandini Reddy,42,42,0,1.000000
59,Manav Sharma,37,37,0,1.000000
100,Riya Gupta,34,34,0,1.000000
28,Divya Fernandes,33,33,0,1.000000
78,Neha Rana,28,28,0,1.000000
47,Karan Verma,21,21,0,1.000000
73,Neha Iyer,9,6,3,0.666667
109,Sameer Fernandes,9,5,4,0.555556
77,Neha Nair,9,7,2,0.777778


In [ ]:
df.columns


Index(['website', 'link', 'unique_id', 'title', 'publish_date', 'content',
       'top_image', 'image_links', 'links_in_text', 'bold_text', 'tweet_id',
       'video', 'topic', 'tags', 'claim', 'investigation', 'topic_parsed',
       'tags_parsed', 'topic_flat', 'label_v1', 'final_label', 'label_binary',
       'user'],
      dtype='object')

In [16]:
user_stats.columns

Index(['user', 'total_posts', 'fake_posts', 'real_posts', 'fake_ratio'], dtype='object')

## Data imbalance correction

In [21]:
india_today  = pd.read_excel("india_today_english.xlsx")
vishwas = pd.read_excel("vishwas_english.xlsx")
factly = pd.read_excel("factly_english.xlsx")

extra_raw  = pd.concat([india_today, vishwas, factly], ignore_index = True)

extra_real_df = extra_raw.reindex(columns=df.columns)

extra_real_df.shape
extra_real_df.tail(3)

,website,link,unique_id,title,publish_date,content,top_image,image_links,links_in_text,bold_text,...,tags,claim,investigation,topic_parsed,tags_parsed,topic_flat,label_v1,final_label,label_binary,user
2010,factly,https://factly.in/no-this-isnt-the-video-of-haryana-farmers-protesting-against-modi-government/,no-this-isnt-the-video-of-haryana-farmers-protesting-against-modi-government/,"No, this isn’t the video of Haryana farmers protesting against Modi government",04-10-2019 00:00:00,['A video is being shared on Facebook with a claim that Haryana farmers are performing last rites to Modi as a sign of protest against his governm...,NaN,"['https://factly.in/wp-content/uploads//2019/10/Haryana-teachers-protests-Fb-post-1024x634.jpg', 'https://factly.in/wp-content/uploads//2019/10/Ha...","['https://www.facebook.com/FanofDChautala/videos/387549088605574/?v=387549088605574', 'https://www.youtube.com/watch?v=7AHuGynVmjw', 'https://time...","['video', 'video', 'article']",...,"['English', 'Fake News']",Claim: Video of Haryana farmers performing last rites to Modi as a sign of protest against his government. A video is being shared on Facebook wit...,"The Facebook post was made on September 30, 2019. In the comments section of the post, a user has commented that the video shows an incident that ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2011,factly,https://factly.in/old-images-are-being-shared-with-a-false-narrative-of-62-degrees-temperature-in-kuwait/,old-images-are-being-shared-with-a-false-narrative-of-62-degrees-temperature-in-kuwait/,Old images are being shared with a false narrative of ‘62-degrees temperature in Kuwait’,17-06-2019 00:00:00,"['On Facebook, a few posts with claims that Kuwait has recorded the world’s highest temperature of 62 degrees Celsius this Saturday are going vira...",NaN,"['https://factly.in/wp-content/uploads//2019/06/62-degrees-temperature-FB-Post.jpg', 'https://factly.in/wp-content/uploads//2019/06/62-degrees-Cel...","['https://www.facebook.com/permalink.php?story_fbid=2424004844553394&id=100008317061496', 'https://news.kuwaittimes.net/website/62-degrees-centigr...","['posts', 'article', 'fact-check article', 'fact-check article', ' tweet']",...,"['English', 'Fake News']","Claim: Kuwait has recorded the world’s highest temperature of 62 degrees Celsius this Saturday. On Facebook, a few posts with claims that Kuwa...","The news that Kuwait has recorded a temperature of 62 degrees Celsius has been doing rounds for the past 3 years. So, in July 2017, “Kuwait Times”...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012,factly,https://factly.in/the-photo-is-neither-related-to-australia-nor-any-wildfire/,the-photo-is-neither-related-to-australia-nor-any-wildfire/,The photo is neither related to Australia nor any wildfire,08-01-2020 00:00:00,['A photo is being shared on social media with a claim that it is related to Australian bushfires. Let’s try to analyze the claim made in the post...,NaN,"['https://factly.in/wp-content/uploads//2020/01/Helicopter-view-of-Aus-Bushfires-FB-Post.jpg', 'https://factly.in/wp-content/uploads//2020/01/Heli...","['https://www.facebook.com/TDM777/photos/a.1231206397018052/1577857062352982/?type=3&theater', 'https://www.reddit.com/r/oblivion/comments/953spg/...","['photo', 'uploaded', 'tweeted', 'article', 'profile']",...,"['English', 'Fake News']",Claim: Helicopter view of Australian bushfires. A photo is being shared on social media with a claim that it is related to Australian bushfires. L...,"When the photo was run through the Yandex Reverse Image Search, the same photo was found to be uploaded by many users as related to ‘California wi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
## feature and label extraction:
import re

def tags_to_flat(tags_val):
    if pd.isna(tags_val):
        return ''
    return str(tags_val).lower()


extra_real_df['tags_flat_check'] = extra_raw['tags'].apply(tags_to_flat) if 'tags' in extra_raw.columns else ''

def derive_label_extra(tags_str):
    tags_str = tags_str.lower()
    if 'fake news' in tags_str:
        return 'fake'
    elif 'fact check' in tags_str or 'factcheck' in tags_str:
        return 'factcheck_ambiguous'
    else:
        return 'other'

extra_real_df['label_v1'] = extra_real_df['tags_flat_check'].apply(derive_label_extra)
print(extra_real_df['label_v1'].value_counts())
print(extra_real_df.groupby(extra_raw['website'])['label_v1'].value_counts())    

label_v1
other                  998
fake                   956
factcheck_ambiguous     59
Name: count, dtype: int64
website          label_v1           
factly           fake                   951
                 factcheck_ambiguous     20
indiaToday       other                  788
vishwas_english  other                  210
                 factcheck_ambiguous     39
                 fake                     5
Name: count, dtype: int64


In [23]:
# Apply the same final_label / label_binary mapping as your original df
extra_real_df['final_label'] = extra_real_df['label_v1'].replace({'factcheck_ambiguous': 'other'})
extra_real_df['label_binary'] = extra_real_df['final_label'].map({'fake': 'fake', 'other': 'real'})

print(extra_real_df['label_binary'].value_counts())

# Keep only the real-labelled rows for this rebalancing step
extra_real_only = extra_real_df[extra_real_df['label_binary'] == 'real'].copy()
print(extra_real_only.shape)

label_binary
real    1057
fake     956
Name: count, dtype: int64
(1057, 24)


In [24]:
import re

def title_verdict(title):
    if pd.isna(title):
        return 'other'
    t = str(title).lower()
    
    fake_signals = [
        r'\bis fake\b', r'\bfake post\b', r'\bfake news\b', r'\bfalse claim\b',
        r'\bis false\b', r'\bmorphed\b', r'\bmisleading\b', r'\bnot true\b',
        r'\bbusted\b', r'\bdebunk', r'^no,', r'\bnot from\b', r'\bnot .*but\b'
    ]
    for pat in fake_signals:
        if re.search(pat, t):
            return 'fake'
    
    # Question-style titles with no clear verdict signal
    if t.strip().endswith('?') or t.startswith('did ') or t.startswith('can '):
        return 'ambiguous'
    
    return 'other'

extra_real_only['title_verdict'] = extra_real_only['title'].apply(title_verdict)
print(extra_real_only['title_verdict'].value_counts())

title_verdict
other        656
fake         264
ambiguous    137
Name: count, dtype: int64


In [25]:
# Flip the 264 rows where the title gives a strong, unambiguous fake signal
strong_fake_mask = extra_real_only['title_verdict'] == 'fake'

extra_real_only.loc[strong_fake_mask, 'label_v1'] = 'fake'
extra_real_only.loc[strong_fake_mask, 'final_label'] = 'fake'
extra_real_only.loc[strong_fake_mask, 'label_binary'] = 'fake'

print(extra_real_only['label_binary'].value_counts())

label_binary
real    793
fake    264
Name: count, dtype: int64


In [27]:
extra_real_only.head(20)

,website,link,unique_id,title,publish_date,content,top_image,image_links,links_in_text,bold_text,...,investigation,topic_parsed,tags_parsed,topic_flat,label_v1,final_label,label_binary,user,tags_flat_check,title_verdict
0,indiaToday,https://www.indiatoday.in/fact-check/story/viral-test-jio-institute-modi-mukesh-ambani-congress-google-institution-of-eminence-1282213-2018-07-10,viral-test-jio-institute-modi-mukesh-ambani-congress-google-institution-of-eminence-1282213-2018-07-10,Viral Test: Why a Google search for Jio Institute is not a good idea,10-07-2018 00:00:00,"[' Jump to navigation', 'Speak Now', ""All hell broke loose on Monday evening when the government announced that Jio Institute has been shortlisted...",NaN,['https://akm-img-a-in.tosshub.com/indiatoday/images/story/201807/Jio_Institute_Eminence_Google_Search_Narendra_Modi_0.jpeg?SO_8rcg1O5RBwZl1vfsUwS...,"['#main-menu', '/news-analysis/story/coronavirus-pandemic-lockdown-new-covid19-cases-states-districts-affected-1672735-2020-04-30', '/india/story/...","['Jump to navigation', 'Covid-19 has spread to 42 new districts since April 17. What did lockdown achieve then?', 'Covid-19 Tracker: State-wise da...",...,Our Viral Test found that the people raising this question have not done their homework properly.Jio Institute became the most trending topic on s...,NaN,NaN,NaN,other,other,real,NaN,,other
1,indiaToday,https://www.indiatoday.in/fact-check/story/fact-check-false-claims-take-off-after-indonesian-air-crash-1379174-2018-10-31,fact-check-false-claims-take-off-after-indonesian-air-crash-1379174-2018-10-31,Fact Check: False claims take off after Indonesian air crash,31-10-2018 00:00:00,"[' Jump to navigation', 'Speak Now', 'Several photos and videos surfaced on social media within few hours of the tragic crash of Lion Air flight J...",NaN,"['https://akm-img-a-in.tosshub.com/indiatoday/images/story/201810/lion_air.jpeg?svqsmu95IKf5zVAk_14RWWt7cP83MOkm', 'https://akm-img-a-in.tosshub.c...","['#main-menu', 'https://www.facebook.com/groups/fafhh/permalink/770951559904048/?__tn__=K-R', 'https://web.archive.org/web/20181030063454/https://...","['Jump to navigation', 'public group', 'archive link', 'pic.twitter.com/SziiS6Uoog', 'pic.twitter.com/VbGYaBlVop', 'BNPB', 'Fact Check: Rahul Gand...",...,India Today Fact Check team found that all these claims are false and have no relation with the recent Lion aircraft crash. A post on Facebook wit...,NaN,NaN,NaN,other,other,real,NaN,,other
2,indiaToday,https://www.indiatoday.in/fact-check/story/fact-check-no-rahul-gandhi-has-not-retracted-from-the-promise-of-loan-waiver-1408511-2018-12-13,fact-check-no-rahul-gandhi-has-not-retracted-from-the-promise-of-loan-waiver-1408511-2018-12-13,"Fact Check: No, Rahul Gandhi has not retracted from the promise of loan waiver",13-12-2018 00:00:00,['The issue of farm loan waiver is said to be one of the main factors which contributed to the victory of Congress in assembly elections. But is C...,NaN,['https://akm-img-a-in.tosshub.com/indiatoday/images/story/201812/rahul_gandhi_1_twitter_bhupesh_baghel_2.jpeg?kr1q8FPpb.vIQmXA3Axjjr2GYtYWAkPw'],"['#main-menu', 'https://www.facebook.com/NationWithNaMo2019/videos/vb.670490466309796/304388916870737/?type=2&theater', 'https://www.facebook.com/...","['Jump to navigation', 'The Nation with NaMo page', 'loan waiver is not the solution', 'seen here', 'Covid-19 has spread to 42 new districts since...",...,"India Today's Fact Check found the ModiNama post to be misleading.In his election campaigns, Rahul had promised to waive off farmers\' loan within...",NaN,NaN,NaN,other,other,real,NaN,,other
3,indiaToday,https://www.indiatoday.in/fact-check/story/fact-check-this-is-not-bruce-lee-playing-ping-pong-with-nunchaku-1616947-2019-11-08,fact-check-this-is-not-bruce-lee-playing-ping-pong-with-nunchaku-1616947-2019-11-08,Fact Check: This is not Bruce Lee playing ping-pong with Nunchaku,08-11-2019 00:00:00,['Many people on social media are astounded and amused by a vide